# ⚡ Aegis Trading AI — GPU Training on Colab (v3)

Train the hybrid TCN→Transformer model (with **candlestick pattern features** from *The Candlestick Trading Bible*) on a free GPU, backtest out-of-sample, and **sweep timeframes to hunt for real signal**.

**Setup:** `Runtime → Change runtime type → GPU (T4)`.

**Stop a cell:** `Runtime → Interrupt execution` (⏹). v3 runs in-process, so interrupt works cleanly.

> Decision-support research, not financial advice. A model that trains is not a model that earns — trust the accuracy sweep, not the training loss.

## 1. Confirm the GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable via Runtime > Change runtime type')

## 2. Upload the project

Run the cell, pick the **latest `aegis_project.zip`** (has candlestick features + the sweep).

In [ ]:
from google.colab import files
import zipfile, os, glob

uploaded = files.upload()  # choose aegis_project.zip
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall('/content/aegis')
root = next(os.path.dirname(p) for p in glob.glob('/content/aegis/**/pyproject.toml', recursive=True))
os.chdir(root)
print('Project root:', root)

## 3. Install the light deps

In [ ]:
!pip install -q pydantic pydantic-settings httpx websockets rich tensorboard nest_asyncio
import logging; logging.getLogger().setLevel(logging.INFO)
print('deps ready')

## 4. 🔎 Signal sweep — hunt for a predictable setup (START HERE)

Trains a model for **every timeframe × horizon** and measures out-of-sample directional accuracy on a big sample. The fastest way to find whether *any* configuration has real signal, instead of polishing one dead end.

Prints a ranked table. Read it honestly: **~50% = noise · 52–54% = faint · ≥55% = worth a real backtest · ≥58% = notable (rare)**. Runs on the GPU; ~20–40 min for the full grid. Shrink `timeframes`/`horizons` to go faster. Interruptible (⏹).

In [ ]:
import nest_asyncio; nest_asyncio.apply()
from app.backtest.sweep import run_sweep

try:
    results = run_sweep(
        symbol='BTCUSDT',
        timeframes=('15m', '1h', '4h', '1d'),
        horizons=(1, 3, 6),
        seq_len=64, epochs=15, folds=3, stride=3,
    )
except KeyboardInterrupt:
    print('\nSweep interrupted — partial results (if any) printed above.')

## 5. (Optional) Train + backtest ONE config in depth

Once the sweep points at a promising timeframe/horizon, train that one properly and cost-aware backtest it. Saves to `artifacts/model_best.pt`. In-process, live progress, interruptible.

In [ ]:
import nest_asyncio; nest_asyncio.apply()
from app.backtest.run import build_parser, main

args = build_parser().parse_args([
    '--symbol', 'BTCUSDT', '--interval', '1h',   # <- set to the sweep's best timeframe
    '--bars', '30000', '--holdout', '0.25',
    '--seq-len', '128', '--horizon', '6',        # <- and its best horizon
    '--epochs', '40', '--folds', '4',
    '--out', 'artifacts/model_best.pt',
])
try:
    summary = main(args)
    print('\nDONE — summary:', summary)
except KeyboardInterrupt:
    print('\nInterrupted cleanly.')

## 6. Download the trained model

Pull `model_best.pt` back and drop it in your local `artifacts/` folder — the app/dashboard use it automatically.

In [ ]:
from google.colab import files
files.download('artifacts/model_best.pt')